# CNN for images

In [1]:
import numpy as np
import pandas as pd

### Load split data

In [2]:
train_df = pd.read_csv(
    "data/train_df.csv",
    parse_dates=["publish_timestamp"]  # converts datestime string into datetime object
)
test_df = pd.read_csv(
    "data/test_df.csv",
    parse_dates=["publish_timestamp"]
)

In [3]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1471 entries, 0 to 1470
Data columns (total 17 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   user_id                   1471 non-null   int64         
 1   following                 1471 non-null   int64         
 2   publish_timestamp         1471 non-null   datetime64[ns]
 3   has_location              1471 non-null   int64         
 4   is_carousel               1471 non-null   int64         
 5   num_images                1471 non-null   int64         
 6   is_sponsored              1471 non-null   int64         
 7   image_path                1471 non-null   object        
 8   caption                   1443 non-null   object        
 9   follower_following_ratio  1471 non-null   float64       
 10  hour                      1471 non-null   int64         
 11  day                       1471 non-null   object        
 12  is_weekend          

### Obtain images

In [ ]:
train_df['image_path'].value_counts()

image_path
.\datasets\Data\hardikpandya93_25298326_3015497697586924812_1434924_4335\2023-01-14_11-22-39_UTC.jpg    1
.\datasets\Data\minimalistbaker_2144759_3038737068927423898_20043_212\2023-02-15_12-55-08_UTC_1.jpg     1
.\datasets\Data\minimalistbaker_2144759_3035852094800723235_1939_20\2023-02-11_13-23-13_UTC_1.jpg       1
.\datasets\Data\minimalistbaker_2144759_3030776337531524399_1468_14\2023-02-04_13-18-35_UTC_1.jpg       1
.\datasets\Data\minimalistbaker_2144759_3030046787029914286_2685_25\2023-02-03_13-09-06_UTC_1.jpg       1
                                                                                                       ..
.\datasets\Data\brendonburchard_1158797_3049912335603423714_2217_98\2023-03-02_22-58-24_UTC_1.jpg       1
.\datasets\Data\brendonburchard_1158797_3048346918890877380_2014_331\2023-02-28_19-08-12_UTC_1.jpg      1
.\datasets\Data\brendonburchard_1158797_3046992068581459508_5934_123\2023-02-26_22-16-21_UTC.jpg        1
.\datasets\Data\brendonburchard_115

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

class ImageDataset(Dataset):
    def __init__(self, df, img_col="image_path", target_col="engagement_label", transform=None):
        self.df = df.reset_index(drop=True)
        self.img_col = img_col
        self.target_col = target_col
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, self.img_col]
        img = Image.open(img_path).convert("RGB")  # make sure it's RGB
        if self.transform:
            img = self.transform(img)
        label = self.df.loc[idx, self.target_col]
        return img, label


In [ ]:
# Standard transforms for CNN
image_size = 128  # resize all images to 128x128
train_transforms = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])


In [ ]:
train_ds = ImageDataset(train_df, transform=train_transforms)
test_ds  = ImageDataset(test_df, transform=test_transforms)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=64, shuffle=False)


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)  # input 3 channels (RGB)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.pool = nn.MaxPool2d(2,2)
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(128 * 16 * 16, 256)  # adjust if image_size changes
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = F.relu(self.conv3(x))
        x = self.pool(x)  # final size (batch, 128, 16,16) if image_size=128
        x = x.view(x.size(0), -1)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SimpleCNN(num_classes=3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


### Pretrained ResNet18 for images

In [ ]:
import torchvision.models as models
import torch.nn.functional as F

resnet = models.resnet18(pretrained=True)

class ImageResNet(nn.Module):
    def __init__(self, resnet_model, num_classes=3, dropout=0.5, freeze_resnet=True):
        super().__init__()
        self.resnet = resnet_model
        if freeze_resnet:
            for param in self.resnet.parameters():
                param.requires_grad = False
        
        # Replace the final fc layer
        self.resnet.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.resnet.fc.in_features, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )
    
    def forward(self, x):
        return self.resnet(x)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = ImageResNet(resnet, num_classes=3).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [ ]:
# device = "cuda" if torch.cuda.is_available() else "cpu" # use GPU if available

# # Input size matching one-hot expanded features
# input_dim = X_train_proc.shape[1]

# # Instantiate model, move weights to device
# model = MetadataMLP(input_dim).to(device)

# # Set loss function and optimizer
# criterion = nn.CrossEntropyLoss() # multiclass classification, labels are integers, logits are raw scores
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-3) # learning rate 0.001 is standard

# training loop
def train_epoch(loader):
    # Put model in training mode
    model.train()

    # Initialize accumulator for total loss
    total_loss = 0

    # Loops over batches in training set
    for X, y in loader:
        # Moves x y to same device
        X, y = X.to(device), y.to(device)

        # Clear old gradients from previous batch
        optimizer.zero_grad()

        # Feeds batch into model, perform forward pass, output logits
        logits = model(X)

        # Compute loss for this batch
        loss = criterion(logits, y)

        # Performs backpropagation: computes gradients of the loss with respect to each model parameter.
        loss.backward()

        # Updates model parameters using those gradients
        optimizer.step()

        # Adds the numeric loss value (converted from a tensor with .item()) to the total loss accumulator.
        total_loss += loss.item()

    # Average loss = total loss / number of batches
    avg_loss = total_loss / len(loader)
    
    return avg_loss




In [ ]:
def test_epoch(loader):
    # Set model to evaluation mode. Disables dropout, freezes batchnorm stats
    model.eval()

    # Initialize accumulator for total loss
    total_loss = 0

    with torch.no_grad(): # Disables gradient tracking inside this block.
    # This saves memory and time, since we don’t need gradients for validation.
    # It also ensures the model’s weights won’t accidentally be updated.
        for X, y in loader:
            X, y = X.to(device), y.to(device)
 
            logits = model(X)

            total_loss += criterion(logits, y).item()

    # Compute average loss
    avg_loss = total_loss / len(loader)
    
    return avg_loss

In [ ]:
from sklearn.metrics import confusion_matrix, f1_score

def evaluate_metrics(loader, model):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)

            logits = model(X)
            preds = logits.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    # Accuracy
    correct = sum(p == y for p, y in zip(all_preds, all_labels))
    accuracy = correct / len(all_labels)

    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)

    # Macro F1
    macro_f1 = f1_score(all_labels, all_preds, average="macro")

    return accuracy, cm, macro_f1


In [ ]:
for epoch in range(10):
    train_loss = train_epoch(train_loader)
    test_loss  = test_epoch(test_loader)

    # Metrics
    train_acc, train_cm, train_macro_f1 = evaluate_metrics(train_loader, model)
    test_acc, test_cm, test_macro_f1 = evaluate_metrics(test_loader, model)

    print(f"\nEpoch {epoch+1}")
    print(f"Train   | loss={train_loss:.4f}, acc={train_acc:.3f}, macro-F1={train_macro_f1:.3f}")
    print(f"Test    | loss={test_loss:.4f}, acc={test_acc:.3f}, macro-F1={test_macro_f1:.3f}")

    print("Train Confusion Matrix:")
    print(train_cm)

    print("Test Confusion Matrix:")
    print(test_cm)



Epoch 1
Train   | loss=1.1812, acc=0.421, macro-F1=0.421
Valid   | loss=1.0886, acc=0.375, macro-F1=0.373
Train Confusion Matrix:
[[221 150  91]
 [160 181 155]
 [153 142 218]]
Validation Confusion Matrix:
[[58 58 31]
 [39 45 30]
 [42 35 38]]

Epoch 2
Train   | loss=1.0861, acc=0.430, macro-F1=0.406
Valid   | loss=1.0870, acc=0.362, macro-F1=0.355
Train Confusion Matrix:
[[132 104 226]
 [ 86 140 270]
 [ 55  98 360]]
Validation Confusion Matrix:
[[33 38 76]
 [23 39 52]
 [26 25 64]]

Epoch 3
Train   | loss=1.0738, acc=0.453, macro-F1=0.438
Valid   | loss=1.0775, acc=0.388, macro-F1=0.374
Train Confusion Matrix:
[[127 235 100]
 [ 66 337  93]
 [ 44 266 203]]
Validation Confusion Matrix:
[[30 82 35]
 [20 77 17]
 [19 57 39]]

Epoch 4
Train   | loss=1.0586, acc=0.483, macro-F1=0.472
Valid   | loss=1.0619, acc=0.428, macro-F1=0.406
Train Confusion Matrix:
[[326  68  68]
 [216 150 130]
 [177 101 235]]
Validation Confusion Matrix:
[[88 26 33]
 [54 29 31]
 [50 21 44]]

Epoch 5
Train   | loss=1.03